In [0]:
print(spark.version)
dbutils.fs.ls("/Volumes/workspace/default/tenday")

4.2.0


[]

In [0]:
"""
Tenday -- Prospect Acquisition & Gaming Prevention Platform
Synthetic data generator.

Runs in PySpark so generation is distributed; scale is set by N_CUSTOMERS.

Produces five Delta tables:

    campaigns       one row per acquisition campaign (offer terms, cost)
    customers       one row per customer, ACQUISITION-TIME attributes only
    transactions    one row per transaction  <- the big table
    redemptions     bonus qualification and redemption events
    hidden_labels   latent archetype per customer  <- EVALUATION ONLY

Design notes
------------
1. Five latent behavioural archetypes drive generation. The models must
   rediscover them. Boundaries are deliberately fuzzy.

2. Gamer propensity is weakly conditioned on acquisition-time attributes
   (channel, offer richness, income band). This gives the Layer 1 prospect
   model a real but modest signal -- which is the intended finding:
   pre-acquisition data explains little, early behaviour explains a lot.

3. `customers` contains NOTHING behavioural. Horizon separation starts here.

4. hidden_labels is ground truth. Never join it into the feature table.
"""

from pyspark.sql import SparkSession, functions as F, Window

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------

N_CUSTOMERS = 200_000
SEED = 42
OUTPUT_PATH = "/Volumes/workspace/default/tenday"
HORIZON_DAYS = 365
WRITE_FORMAT = "delta"              # "parquet" for plain local Spark

# (name, share, txn_lo, txn_hi, log_amt_mu, log_amt_sigma, cat_focus, day_skew)
ARCHETYPES = [
    ("steady_builder",  0.34,  90, 260, 6.5, 0.85, 0.28, "uniform"),
    ("high_value",      0.14, 180, 520, 7.4, 0.95, 0.20, "growing"),
    ("occasional",      0.24,  12,  70, 6.2, 0.80, 0.38, "uniform"),
    ("early_churner",   0.16,  15,  90, 6.4, 0.85, 0.44, "front"),
    ("bonus_gamer",     0.12,  25, 120, 6.9, 1.10, 0.62, "sprint"),
]
BASE_GAMER_RATE = 0.12

CATEGORIES = [
    "grocery", "dining", "fuel", "travel", "electronics",
    "apparel", "utilities", "entertainment", "pharmacy",
    "home", "gift_cards", "money_transfer", "wholesale_club",
]

# Acquisition-time attributes. The numbers are gamer-propensity ADJUSTMENTS
# (additive, on top of BASE_GAMER_RATE). Kept small on purpose: the prospect
# layer should be weakly predictive, not a giveaway.
ACQ_CHANNELS = {
    "paid_search":  +0.045,
    "affiliate":    +0.075,
    "direct_mail":  +0.010,
    "branch":       -0.055,
    "referral":     -0.050,
}
INCOME_BANDS = {"low": +0.020, "mid": 0.000, "high": -0.025}
AGE_BANDS = ["18-25", "26-35", "36-50", "51-65", "65+"]
REGIONS = ["north", "south", "east", "west", "central"]

# campaign_id, type, offer, bonus_amount, minimum_spend, acq_cost, gamer_adj
CAMPAIGNS = [
    ("C001", "welcome_bonus", "cashback_5k",   5000,  60000, 1200, +0.030),
    ("C002", "welcome_bonus", "cashback_10k", 10000, 120000, 1800, +0.070),
    ("C003", "welcome_bonus", "points_40k",   12000, 150000, 2200, +0.085),
    ("C004", "fee_waiver",    "fee_waiver",    2500,  80000,  700, -0.020),
    ("C005", "cashback",      "cashback_5k",   5000,  70000, 1100, +0.020),
    ("C006", "travel",        "points_40k",   12000, 175000, 2600, +0.055),
    ("C007", "retention",     "fee_waiver",    2000,  45000,  500, -0.045),
    ("C008", "referral",      "cashback_5k",   4000,  70000,  600, -0.055),
]


def build_spark():
    return (
        SparkSession.builder
        .appName("tenday_generator")
        .config("spark.sql.shuffle.partitions", "64")
        .getOrCreate()
    )


def pick(values, ucol):
    """Uniform choice from `values` driven by a [0,1) column."""
    n = len(values)
    col = F.when(F.col(ucol) < F.lit(1.0 / n), F.lit(values[0]))
    for i, v in enumerate(values[1:-1], start=1):
        col = col.when(F.col(ucol) < F.lit((i + 1) / n), F.lit(v))
    return col.otherwise(F.lit(values[-1]))


def map_col(colname, mapping, default=0.0):
    """Map a string column to numeric values via a dict."""
    out = F.lit(default)
    for k, v in mapping.items():
        out = F.when(F.col(colname) == F.lit(k), F.lit(v)).otherwise(out)
    return out


# ----------------------------------------------------------------------
# 1. campaigns
# ----------------------------------------------------------------------

def make_campaigns(spark):
    rows = [
        (cid, ctype, offer, bonus, min_spend, cost, adj)
        for cid, ctype, offer, bonus, min_spend, cost, adj in CAMPAIGNS
    ]
    cols = ["campaign_id", "campaign_type", "offer_type", "bonus_amount",
            "minimum_spend", "acquisition_cost", "_gamer_adj"]
    return spark.createDataFrame(rows, cols)


# ----------------------------------------------------------------------
# 2. customers  (acquisition-time attributes ONLY)
# ----------------------------------------------------------------------

def make_customers(spark, campaigns, n_customers, seed):
    df = spark.range(0, n_customers).withColumnRenamed("id", "customer_id")

    for i, name in enumerate([
        "u_camp", "u_acq", "u_age", "u_income", "u_region", "u_limit",
        "u_gamer", "u_arch", "u_cat", "u_ntxn", "u_noise", "u_signup",
    ], start=1):
        df = df.withColumn(name, F.rand(seed + i))

    # Acquisition-time attributes
    campaign_ids = [c[0] for c in CAMPAIGNS]
    df = (
        df
        .withColumn("campaign_id", pick(campaign_ids, "u_camp"))
        .withColumn("acquisition_channel", pick(list(ACQ_CHANNELS), "u_acq"))
        .withColumn("age_band", pick(AGE_BANDS, "u_age"))
        .withColumn("income_band", pick(list(INCOME_BANDS), "u_income"))
        .withColumn("region", pick(REGIONS, "u_region"))
        .withColumn("acquisition_date",
                    F.date_add(F.lit("2024-01-01").cast("date"),
                               (F.col("u_signup") * 730).cast("int")))
    )

    # Credit limit correlates with income band (realistic, and a usable
    # prospect feature).
    income_mult = map_col("income_band", {"low": 0.55, "mid": 1.0, "high": 1.9}, 1.0)
    df = df.withColumn(
        "credit_limit",
        ((F.lit(80000) + F.col("u_limit") * F.lit(700000)) * income_mult).cast("int"),
    )

    df = df.join(F.broadcast(campaigns), "campaign_id", "left")

    # --- gamer propensity from acquisition-time attributes only ---------
    propensity = (
        F.lit(BASE_GAMER_RATE)
        + map_col("acquisition_channel", ACQ_CHANNELS)
        + map_col("income_band", INCOME_BANDS)
        + F.col("_gamer_adj")
    )
    # Keep it bounded and modest -- weak signal by design.
    propensity = F.least(F.lit(0.32), F.greatest(F.lit(0.02), propensity))
    df = df.withColumn("_p_gamer", propensity)

    # Draw gamer status, then assign the remaining archetypes proportionally.
    others = [(n, s) for n, s, *_ in ARCHETYPES if n != "bonus_gamer"]
    total_other = sum(s for _, s in others)

    arch = F.lit(None).cast("string")
    cum = 0.0
    for name, share in others:
        lo, hi = cum, cum + share / total_other
        arch = F.when(
            (F.col("u_arch") >= F.lit(lo)) & (F.col("u_arch") < F.lit(hi)),
            F.lit(name),
        ).otherwise(arch)
        cum = hi
    arch = F.coalesce(arch, F.lit("steady_builder"))

    df = df.withColumn(
        "archetype",
        F.when(F.col("u_gamer") < F.col("_p_gamer"), F.lit("bonus_gamer"))
         .otherwise(arch),
    )

    # Attach behavioural parameters implied by the archetype.
    def param(idx, cast="double"):
        col = F.lit(None).cast(cast)
        for row in ARCHETYPES:
            col = F.when(F.col("archetype") == F.lit(row[0]),
                         F.lit(row[idx])).otherwise(col)
        return col

    df = (
        df
        .withColumn("_txn_lo",    param(2, "int"))
        .withColumn("_txn_hi",    param(3, "int"))
        .withColumn("_amt_mu",    param(4))
        .withColumn("_amt_sigma", param(5))
        .withColumn("_cat_focus", param(6))
        .withColumn("_day_skew",  param(7, "string"))
        .withColumn("preferred_category", pick(CATEGORIES, "u_cat"))
    )

    df = df.withColumn(
        "n_txn",
        F.greatest(
            F.lit(3),
            (F.col("_txn_lo")
             + F.col("u_ntxn") * (F.col("_txn_hi") - F.col("_txn_lo"))
             + (F.col("u_noise") - 0.5) * 40).cast("int"),
        ),
    )

    df = df.withColumn("acquisition_month",
                       F.date_format("acquisition_date", "yyyy-MM"))

    drop = [c for c in df.columns if c.startswith("u_")]
    return df.drop(*drop)


# ----------------------------------------------------------------------
# 3. transactions
# ----------------------------------------------------------------------

def make_transactions(customers, seed, horizon_days):
    df = customers.withColumn(
        "txn_seq", F.explode(F.sequence(F.lit(1), F.col("n_txn")))
    )

    for i, name in enumerate(
        ["r_day", "r_day2", "r_cat", "r_cat2", "r_chan", "r_hour", "r_merch"],
        start=31,
    ):
        df = df.withColumn(name, F.rand(seed + i))
    df = df.withColumn("r_amt", F.randn(seed + 40))

    frac = F.col("r_day")
    day_frac = (
        F.when(F.col("_day_skew") == "sprint",
               F.pow(frac, F.lit(3.0)) * F.lit(0.16)
               + F.when(F.col("r_day2") > F.lit(0.965), frac * F.lit(0.55))
                .otherwise(F.lit(0.0)))
         .when(F.col("_day_skew") == "front", F.pow(frac, F.lit(2.6)) * F.lit(0.45))
         .when(F.col("_day_skew") == "growing", F.pow(frac, F.lit(0.55)))
         .otherwise(frac)
    )

    df = df.withColumn(
        "days_since_acquisition",
        F.least(F.lit(horizon_days - 1), (day_frac * F.lit(horizon_days)).cast("int")),
    )
    df = df.withColumn(
        "txn_ts",
        (F.date_add(F.col("acquisition_date"), F.col("days_since_acquisition"))
         .cast("timestamp")
         + F.expr("INTERVAL 1 HOUR") * (F.col("r_hour") * 23).cast("int")),
    )

    df = df.withColumn(
        "amount",
        F.round(
            F.exp(F.col("_amt_mu") + F.col("r_amt") * F.col("_amt_sigma"))
            * F.when((F.col("_day_skew") == "sprint")
                     & (F.col("r_cat2") > F.lit(0.82)), F.lit(6.5))
               .otherwise(F.lit(1.0)),
            2,
        ),
    )
    df = df.withColumn("amount", F.least(F.col("amount"), F.lit(900000.0)))

    n = len(CATEGORIES)
    other_cat = F.when(F.col("r_cat2") < F.lit(1.0 / n), F.lit(CATEGORIES[0]))
    for i, c in enumerate(CATEGORIES[1:-1], start=1):
        other_cat = other_cat.when(F.col("r_cat2") < F.lit((i + 1) / n), F.lit(c))
    other_cat = other_cat.otherwise(F.lit(CATEGORIES[-1]))

    liquid = (F.when(F.col("r_cat2") < F.lit(0.45), F.lit("gift_cards"))
               .when(F.col("r_cat2") < F.lit(0.75), F.lit("money_transfer"))
               .otherwise(F.lit("wholesale_club")))

    df = df.withColumn(
        "merchant_category",
        F.when(F.col("r_cat") < F.col("_cat_focus"), F.col("preferred_category"))
         .when((F.col("_day_skew") == "sprint") & (F.col("r_cat") < F.lit(0.9)), liquid)
         .otherwise(other_cat),
    )

    df = (
        df
        .withColumn("channel",
                    F.when(F.col("r_chan") < F.lit(0.55), F.lit("online"))
                     .when(F.col("r_chan") < F.lit(0.9), F.lit("pos"))
                     .otherwise(F.lit("recurring")))
        .withColumn("merchant_id",
                    F.concat(F.lit("M"),
                             F.lpad((F.col("r_merch") * 4999).cast("int")
                                    .cast("string"), 4, "0")))
        .withColumn("transaction_id",
                    F.concat_ws("-", F.col("customer_id").cast("string"),
                                F.col("txn_seq").cast("string")))
    )

    return df.select(
        "transaction_id", "customer_id", "txn_ts", "days_since_acquisition",
        "amount", "merchant_category", "merchant_id", "channel",
        "acquisition_month",
    )


# ----------------------------------------------------------------------
# 4. data quality issues
# ----------------------------------------------------------------------

def add_data_quality_issues(txns, seed):
    txns = txns.withColumn("q", F.rand(seed + 51))
    txns = txns.withColumn(
        "amount",
        F.when(F.col("q") < F.lit(0.004), F.lit(None).cast("double"))
         .when((F.col("q") >= F.lit(0.004)) & (F.col("q") < F.lit(0.0055)),
               -F.col("amount"))
         .otherwise(F.col("amount")),
    )
    txns = txns.withColumn(
        "merchant_category",
        F.when((F.col("q") >= F.lit(0.0055)) & (F.col("q") < F.lit(0.0075)),
               F.lit(None).cast("string"))
         .otherwise(F.col("merchant_category")),
    ).drop("q")

    dupes = txns.sample(False, 0.001, seed + 52)
    return txns.unionByName(dupes)


# ----------------------------------------------------------------------
# 5. redemptions  (derived from actual spend vs campaign minimum)
# ----------------------------------------------------------------------

def make_redemptions(customers, txns, seed):
    """Qualification = cumulative spend crossing the campaign minimum.
    Redemption lag is behavioural: gamers redeem almost immediately."""

    cust = customers.select(
        "customer_id", "campaign_id", "acquisition_date",
        "minimum_spend", "bonus_amount", "archetype",
    )

    t = (txns.filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
             .select("customer_id", "days_since_acquisition", "amount")
             .join(F.broadcast(cust.select("customer_id", "minimum_spend")),
                   "customer_id"))

    w = (Window.partitionBy("customer_id")
               .orderBy("days_since_acquisition")
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))
    t = t.withColumn("cum_spend", F.sum("amount").over(w))

    qualified = (
        t.filter(F.col("cum_spend") >= F.col("minimum_spend"))
         .groupBy("customer_id")
         .agg(F.min("days_since_acquisition").alias("days_to_qualify"))
    )

    r = (cust.join(qualified, "customer_id", "left")
             .withColumn("u_lag", F.rand(seed + 61))
             .withColumn("u_redeem", F.rand(seed + 62)))

    # Gamers redeem fast; everyone else drifts.
    lag = (F.when(F.col("archetype") == "bonus_gamer",
                  (F.col("u_lag") * 6).cast("int"))
            .otherwise((F.lit(7) + F.col("u_lag") * 80).cast("int")))

    # Not everyone who qualifies bothers to redeem.
    redeem_p = F.when(F.col("archetype") == "bonus_gamer", F.lit(0.99)) \
                .otherwise(F.lit(0.72))

    r = (
        r
        .withColumn("qualified", F.col("days_to_qualify").isNotNull())
        .withColumn("bonus_redeemed",
                    F.col("qualified") & (F.col("u_redeem") < redeem_p))
        .withColumn("redemption_lag_days",
                    F.when(F.col("bonus_redeemed"), lag))
        .withColumn("days_to_redeem",
                    F.col("days_to_qualify") + F.col("redemption_lag_days"))
        .withColumn("bonus_earned",
                    F.when(F.col("bonus_redeemed"), F.col("bonus_amount"))
                     .otherwise(F.lit(0)))
        .withColumn("offer_received_date", F.col("acquisition_date"))
        .withColumn("qualification_date",
                    F.date_add(F.col("acquisition_date"), F.col("days_to_qualify")))
        .withColumn("redemption_date",
                    F.date_add(F.col("acquisition_date"), F.col("days_to_redeem")))
    )

    return r.select(
        "customer_id", "campaign_id", "offer_received_date",
        "qualified", "days_to_qualify", "qualification_date",
        "bonus_redeemed", "redemption_date", "redemption_lag_days",
        "bonus_earned",
    )


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------

def main(spark=None, out=None, n_customers=None, write_format=None):
    """Serverless-safe: no .cache(), no spark.stop().

    Databricks serverless does not allow PERSIST, so each stage is written
    to storage and read back instead. That materialises the data the same
    way caching would, and has the side benefit of checkpointing every
    stage -- a failure at step 4 does not force a regeneration of step 2.
    """
    if spark is None:
        spark = build_spark()
    out = out or OUTPUT_PATH
    n_customers = n_customers or N_CUSTOMERS
    fmt = write_format or WRITE_FORMAT

    # 1. campaigns
    campaigns = make_campaigns(spark)
    (campaigns.drop("_gamer_adj").write.format(fmt)
        .mode("overwrite").save(f"{out}/campaigns"))

    # 2. customers -- staged with internal columns, then read back
    (make_customers(spark, campaigns, n_customers, SEED)
        .write.format(fmt).mode("overwrite")
        .save(f"{out}/_staging_customers"))
    customers = spark.read.format(fmt).load(f"{out}/_staging_customers")

    (customers.select(
        "customer_id", "acquisition_date", "acquisition_month",
        "acquisition_channel", "campaign_id", "age_band", "income_band",
        "region", "credit_limit",
     ).write.format(fmt).mode("overwrite").save(f"{out}/customers"))

    (customers.select(
        "customer_id",
        F.col("archetype").alias("true_archetype"),
        (F.col("archetype") == F.lit("bonus_gamer")).cast("int").alias("is_gamer"),
     ).write.format(fmt).mode("overwrite").save(f"{out}/hidden_labels"))

    # 3. transactions
    (add_data_quality_issues(
        make_transactions(customers, SEED, HORIZON_DAYS), SEED)
        .write.format(fmt).mode("overwrite")
        .partitionBy("acquisition_month").save(f"{out}/transactions"))
    txns = spark.read.format(fmt).load(f"{out}/transactions")

    # 4. redemptions
    (make_redemptions(customers, txns, SEED)
        .write.format(fmt).mode("overwrite").save(f"{out}/redemptions"))

    for t in ["campaigns", "customers", "transactions",
              "redemptions", "hidden_labels"]:
        n = spark.read.format(fmt).load(f"{out}/{t}").count()
        print(f"{t:<14} {n:,}")

main()

campaigns      8
customers      200,000
transactions   26,656,314
redemptions    200,000
hidden_labels  200,000


In [0]:
OUT = "/Volumes/workspace/default/tenday"

txns  = spark.read.format("delta").load(f"{OUT}/transactions")
cust  = spark.read.format("delta").load(f"{OUT}/customers")
red   = spark.read.format("delta").load(f"{OUT}/redemptions")
camp  = spark.read.format("delta").load(f"{OUT}/campaigns")

txns.printSchema()
txns.show(10, truncate=False)
cust.show(5, truncate=False)
camp.show(truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- txn_ts: timestamp (nullable = true)
 |-- days_since_acquisition: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- acquisition_month: string (nullable = true)

+--------------+-----------+-------------------+----------------------+-------+-----------------+-----------+---------+-----------------+
|transaction_id|customer_id|txn_ts             |days_since_acquisition|amount |merchant_category|merchant_id|channel  |acquisition_month|
+--------------+-----------+-------------------+----------------------+-------+-----------------+-----------+---------+-----------------+
|175004-1      |175004     |2025-01-29 11:00:00|315                   |1652.42|money_transfer   |M3747      |online   |2024-03          |
|175004-2      |175004     |2024-06-07 00

In [0]:
from pyspark.sql import functions as F

total = txns.count()
print("total rows        ", total)
print("null amounts      ", txns.filter(F.col("amount").isNull()).count())
print("negative amounts  ", txns.filter(F.col("amount") < 0).count())
print("null categories   ", txns.filter(F.col("merchant_category").isNull()).count())
print("distinct txn ids  ", txns.select("transaction_id").distinct().count())

total rows         26656314
null amounts       105920
negative amounts   39950
null categories    53326
distinct txn ids   26629641


In [0]:
txns.select("amount").describe().show()

txns.groupBy("merchant_category").count().orderBy(F.desc("count")).show()

(txns.groupBy("customer_id").count()
     .describe("count").show())

+-------+------------------+
|summary|            amount|
+-------+------------------+
|  count|          26550394|
|   mean| 1702.341887987043|
| stddev|3315.4727649245187|
|    min|        -141335.24|
|    max|          900000.0|
+-------+------------------+

+-----------------+-------+
|merchant_category|  count|
+-----------------+-------+
|       gift_cards|2264535|
|   money_transfer|2172807|
|   wholesale_club|2143093|
|      electronics|2012976|
|    entertainment|2008304|
|             fuel|2005706|
|             home|2005042|
|          grocery|2004456|
|         pharmacy|2000896|
|           travel|2000378|
|           dining|1996799|
|          apparel|1995676|
|        utilities|1992320|
|             NULL|  53326|
+-----------------+-------+

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|            200000|
|   mean|         133.28157|
| stddev|114.01726798174955|
|    min|                 3|
|    max|               539|
+

## Data profile — 200k customers, 26.7M transactions

**Volume**
- 26,656,314 transactions across 200,000 customers
- Mean 133 transactions per customer (min 3, max 539)

**Data quality**
- 105,920 null amounts (0.40%) — dropped, not imputed, to avoid
  corrupting cumulative-spend features
- 39,950 negative amounts (0.15%) — refunds, separated from spend
  aggregates but counted as a behavioural feature
- 53,326 null categories (0.20%) — labelled 'unknown'; row retained
  because the amount is valid
- 26,673 duplicate transaction_ids (0.10%) — deduplicated

**Distribution**
- Mean transaction ₹1,702, stddev ₹3,315 — heavily right-skewed
- Implication: log-transform before scaling for K-Means, or a small
  number of large transactions will dominate cluster boundaries

**First signal, pre-modelling**
- gift_cards, money_transfer and wholesale_club are the three most
  common categories, ahead of grocery, fuel and dining
- These are cash-equivalent categories. Their over-representation is
  consistent with manufactured spend and is visible in a groupBy,
  before any model is fitted